# Actividad 05 - 02. Detectores: Gitleaks (reglas) y Random Forest (aprendizaje automático)

**NewCodeCreds** · Edward Alexander Gómez Ispanel, Jocelyn Abigail De León Ardón y Brazil Abraham
Batres Cruz - Maestría en Data Science - Investigación en Data Science - Noviembre 2026

Datos: la tabla de análisis del notebook 01 (`datos_procesados/lineas.parquet`, una fila por
línea candidata de CredData) y los hallazgos de Gitleaks sobre `CredData/data`.

Genera una predicción fuera de muestra para cada línea candidata y cada partición.

- **Gitleaks 8.30.1** no se entrena: se ejecuta una vez sobre `CredData/data` y una línea
  cuenta como detectada si algún hallazgo cae dentro de su rango de líneas.
- **Random Forest** se reentrena en cada pliegue de cada partición con rasgos de la línea
  (largo, entropía, tipos de caracteres, palabras clave) y n-gramas de caracteres. No usa la
  etiqueta, la categoría, la ruta ni la extensión del archivo.

Gitleaks se corrió desde la terminal (ver `herramientas/`):

`gitleaks dir ../CredData/data -f json -r datos_procesados/gitleaks_hallazgos.json --exit-code 0`

## 0. Configuración

Hiperparámetros de Random Forest: 200 árboles, hojas de al menos 2 líneas, `max_features="sqrt"`,
pesos balanceados por submuestra y umbral 0.5. N-gramas de caracteres de 2 a 4, los 3,000 más
frecuentes del entrenamiento de cada pliegue.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

sys.path.append(".")
from ids_utils import SEMILLA, caracteristicas_manuales

DATOS = Path("../datos_procesados")
RES = Path("../resultados")
N_ARBOLES = 200
MAX_NGRAMAS = 3000
UMBRAL = 0.5
lineas = pd.read_parquet(DATOS / "lineas.parquet")
print(lineas.shape)

(64797, 22)


## 1. Gitleaks: hallazgos -> líneas candidatas

### 1.1 Cruce de hallazgos con las líneas etiquetadas

In [2]:
hallazgos = pd.DataFrame(json.loads((DATOS / "gitleaks_hallazgos.json").read_text()))
hallazgos["FilePath"] = hallazgos.File.str.extract(r"(data/.*)$", expand=False)
print("Hallazgos de Gitleaks:", len(hallazgos), "| reglas distintas:", hallazgos.RuleID.nunique())

por_archivo = {fp: g[["StartLine", "EndLine"]].to_numpy() for fp, g in hallazgos.groupby("FilePath")}


def detectada(fp, ls, le):
    rangos = por_archivo.get(fp)
    if rangos is None:
        return 0
    return int(((rangos[:, 0] <= le) & (rangos[:, 1] >= ls)).any())


lineas["pred_gitleaks"] = [detectada(*t) for t in zip(lineas.FilePath, lineas.LineStart, lineas.LineEnd)]

# Hallazgos que no corresponden a ninguna línea candidata (fuera del conjunto etiquetado)
claves = set(zip(lineas.FilePath, lineas.LineStart))
fuera = sum((fp, sl) not in claves for fp, sl in zip(hallazgos.FilePath, hallazgos.StartLine))
print("Hallazgos fuera de las líneas etiquetadas (aprox.):", fuera)

Hallazgos de Gitleaks: 8350 | reglas distintas: 28


Hallazgos fuera de las líneas etiquetadas (aprox.): 247


### 1.2 Detección de Gitleaks por etiqueta

In [3]:
print(pd.crosstab(lineas.GroundTruth, lineas.pred_gitleaks, margins=True))

pred_gitleaks      0     1    All
GroundTruth                      
F              44385  1112  45497
T               8275  7005  15280
X               3997    23   4020
All            56657  8140  64797


**Qué salió.** Gitleaks detecta 7,005 de las 15,280 líneas T (recall 45.8%) y marca además 1,112
líneas F y 23 X. Sus 8,350 hallazgos vienen de 28 reglas, y 247 caen fuera de las líneas etiquetadas,
así que no se cuentan. El recall bajo es esperable por dos razones. Primero, sus reglas no buscan
contraseñas genéricas, UUID ni autenticación básica, que CredData etiqueta como T. Segundo, corre
sobre los valores ofuscados, que perdieron prefijos como `AKIA` (AWS) o `ghp_` (GitHub), en los que
se apoyan varias de sus reglas.

## 2. Random Forest: matriz de rasgos

In [4]:
manuales = caracteristicas_manuales(lineas.texto)
X_man = sparse.csr_matrix(manuales.to_numpy(dtype=np.float32))
print("Rasgos manuales:", manuales.shape[1])


def entrenar_y_predecir(idx_tr, idx_te, y):
    """Ajusta el vectorizador y el bosque SOLO con entrenamiento; devuelve probabilidades."""
    vec = CountVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=MAX_NGRAMAS,
                          binary=True, lowercase=True, dtype=np.float32)
    X_tr = sparse.hstack([X_man[idx_tr], vec.fit_transform(lineas.texto.iloc[idx_tr])]).tocsr()
    X_te = sparse.hstack([X_man[idx_te], vec.transform(lineas.texto.iloc[idx_te])]).tocsr()
    rf = RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=2, max_features="sqrt",
                                class_weight="balanced_subsample", n_jobs=-1, random_state=SEMILLA)
    rf.fit(X_tr, y[idx_tr])
    return rf.predict_proba(X_te)[:, 1]

Rasgos manuales: 34


## 3. Predicciones fuera de muestra en cada partición

In [5]:
esquemas = {
    "aleatorio_crudo": ("pliegue_aleatorio_crudo", "y"),
    "aleatorio": ("pliegue_aleatorio", "y_dedup"),
    "organizacion": ("pliegue_organizacion", "y_dedup"),
    "tipo": ("pliegue_tipo", "y_dedup"),
}
tiempos = []
for nombre, (col, col_y) in esquemas.items():
    lineas[f"score_rf_{nombre}"] = np.nan
    y = lineas[col_y].fillna(0).to_numpy().astype(int)
    if nombre == "tipo":
        pliegues = [p for p in lineas[col].unique() if p != ""]
        universo = lineas.index[lineas.dedup == 1]       # entrena con todo el conjunto deduplicado
    else:
        pliegues = sorted(p for p in lineas[col].unique() if p != -1)
        universo = lineas.index[lineas[col] != -1]
    for p in pliegues:
        t0 = time.time()
        idx_te = lineas.index[lineas[col] == p].to_numpy()
        if nombre == "tipo":
            # deja fuera TODO el lenguaje, no solo las líneas deduplicadas de prueba
            fuera_tipo = lineas.tipo_archivo == p
            idx_tr = np.setdiff1d(universo, lineas.index[fuera_tipo])
        else:
            idx_tr = np.setdiff1d(universo, idx_te)
        lineas.loc[idx_te, f"score_rf_{nombre}"] = entrenar_y_predecir(idx_tr, idx_te, y)
        tiempos.append({"esquema": nombre, "pliegue": p, "n_train": len(idx_tr),
                        "n_test": len(idx_te), "segundos": round(time.time() - t0, 1)})
        print(tiempos[-1])
    lineas[f"pred_rf_{nombre}"] = (lineas[f"score_rf_{nombre}"] >= UMBRAL).astype("Int64").where(
        lineas[f"score_rf_{nombre}"].notna())

{'esquema': 'aleatorio_crudo', 'pliegue': 0, 'n_train': 51837, 'n_test': 12960, 'segundos': 7.6}


{'esquema': 'aleatorio_crudo', 'pliegue': 1, 'n_train': 51837, 'n_test': 12960, 'segundos': 7.6}


{'esquema': 'aleatorio_crudo', 'pliegue': 2, 'n_train': 51838, 'n_test': 12959, 'segundos': 7.8}


{'esquema': 'aleatorio_crudo', 'pliegue': 3, 'n_train': 51838, 'n_test': 12959, 'segundos': 7.7}


{'esquema': 'aleatorio_crudo', 'pliegue': 4, 'n_train': 51838, 'n_test': 12959, 'segundos': 7.6}


{'esquema': 'aleatorio', 'pliegue': 0, 'n_train': 39703, 'n_test': 9926, 'segundos': 6.3}


... (13 outputs left out)

{'esquema': 'tipo', 'pliegue': 'PHP', 'n_train': 48401, 'n_test': 1228, 'segundos': 7.5}


{'esquema': 'tipo', 'pliegue': 'Java', 'n_train': 48227, 'n_test': 1402, 'segundos': 7.0}


{'esquema': 'tipo', 'pliegue': 'TypeScript', 'n_train': 47910, 'n_test': 1719, 'segundos': 7.0}


{'esquema': 'tipo', 'pliegue': 'Ruby', 'n_train': 47368, 'n_test': 2261, 'segundos': 6.9}


### 3.1 Tiempos de ajuste

In [6]:
tiempos = pd.DataFrame(tiempos)
tiempos.to_csv(RES / "tiempos_random_forest.csv", index=False)
tiempos.groupby("esquema").agg(ajustes=("pliegue", "size"), n_train_medio=("n_train", "mean"),
                               segundos_medio=("segundos", "mean"), segundos_total=("segundos", "sum")).round(1)

,ajustes,n_train_medio,segundos_medio,segundos_total
esquema,,,,
aleatorio,5,39703.2,6.2,31.0
aleatorio_crudo,5,51837.6,7.7,38.3
organizacion,5,39703.2,5.9,29.6
tipo,8,47292.1,7.0,55.7


**Qué salió.** Los 23 ajustes de Random Forest tardaron unos 2.5 minutos en total en el Mac M5 Pro,
unos 6 a 8 segundos cada uno (en la máquina virtual eran unos 70 segundos). Cada ajuste aleatorio
o por organización entrena con unas 39,700 líneas, y cada ajuste por tipo de archivo con unas
47,300, porque deja fuera un solo lenguaje.

## 4. Guardar predicciones (sin el texto de las líneas)

In [7]:
cols_pred = [c for c in lineas.columns if c.startswith(("pred_", "score_"))]
lineas.drop(columns=["texto"]).to_parquet(DATOS / "predicciones.parquet", index=False)
manuales.assign(id_linea=lineas.id_linea.values).to_parquet(DATOS / "rasgos_manuales.parquet", index=False)
lineas[cols_pred].describe().T

,count,mean,std,min,25%,50%,75%,max
pred_gitleaks,64797.0,0.125623,0.331427,0.0,0.0,0.0,0.0,1.0
score_rf_aleatorio_crudo,64797.0,0.256806,0.363087,0.0,0.005954,0.054468,0.373091,1.0
pred_rf_aleatorio_crudo,64797.0,0.230288,0.421021,0.0,0.0,0.0,0.0,1.0
score_rf_aleatorio,49629.0,0.324132,0.386971,0.0,0.017739,0.097584,0.746142,1.0
pred_rf_aleatorio,49629.0,0.299301,0.457956,0.0,0.0,0.0,1.0,1.0
score_rf_organizacion,49629.0,0.334378,0.295474,0.0,0.081429,0.2366,0.570855,0.998751
pred_rf_organizacion,49629.0,0.284229,0.451051,0.0,0.0,0.0,1.0,1.0
score_rf_tipo,18695.0,0.232063,0.272015,0.0,0.038232,0.1131,0.310536,0.99656
pred_rf_tipo,18695.0,0.17486,0.379857,0.0,0.0,0.0,0.0,1.0
